# Predicting CTA L Station Busyness

**Question:** on which days should a tourist expect an L station to be crowded?

**Data:** [CTA Ridership — 'L' Station Entries, Daily Totals](https://data.cityofchicago.org/Transportation/CTA-Ridership-L-Station-Entries-Daily-Totals/5neh-572f)
(Socrata ID `5neh-572f`) — one row per station per day. Restricted to **2022+**: ridership
patterns changed permanently during the pandemic, and training on 2015–2019 would bias
every estimate.

**Approach:**
1. Baseline: each station's mean rides per day-of-week (what the app uses when this notebook hasn't been run).
2. Model: `HistGradientBoostingRegressor` with station, day-of-week, month, week-of-year, and a federal-holiday flag.
3. **Time-based split** — train on 2022–2024, test on 2025+. Random splits leak temporal structure and overstate accuracy.
4. Export `data/ridership_lookup.csv` (station × day-of-week × month predictions with busyness tiers). The Streamlit app automatically prefers this file over the live historical average.

**Honest scope:** the data is *daily* entries, so this predicts which **days** are busy —
not which hours. It cannot distinguish rush hour from midday.

In [ ]:
import io
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OrdinalEncoder

BASE_URL = "https://data.cityofchicago.org/resource/5neh-572f.csv"
START_DATE = "2022-01-01"
SPLIT_DATE = "2025-01-01"          # train before, test after
CACHE_PATH = "../data/ridership_daily.csv"   # raw download cache (gitignored)
LOOKUP_PATH = "../data/ridership_lookup.csv" # consumed by the app

## 1. Download

Paginated pull of the daily totals (~200k rows for 2022+), cached locally so reruns don't hit the API.

In [ ]:
def fetch_daily():
    frames, offset, page = [], 0, 50000
    while True:
        url = (
            f"{BASE_URL}?$select=station_id,stationname,date,daytype,rides"
            f"&$where=date>'{START_DATE}'&$order=date,station_id&$limit={page}&$offset={offset}"
        )
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            chunk = pd.read_csv(io.StringIO(resp.read().decode()))
        if len(chunk) == 0:
            break
        frames.append(chunk)
        offset += page
        if len(chunk) < page:
            break
    return pd.concat(frames, ignore_index=True)


if os.path.exists(CACHE_PATH):
    df = pd.read_csv(CACHE_PATH, parse_dates=["date"])
else:
    df = fetch_daily()
    df["date"] = pd.to_datetime(df["date"])
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    df.to_csv(CACHE_PATH, index=False)

df["station_id"] = df["station_id"].astype(str)
print(f"{len(df):,} rows | {df['station_id'].nunique()} stations | {df['date'].min():%Y-%m-%d} to {df['date'].max():%Y-%m-%d}")

## 2. A quick look

Systemwide ridership: strong weekly seasonality, summer peaks, holiday dips — exactly the structure the features below target.

In [ ]:
daily_total = df.groupby("date")["rides"].sum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_total.index, daily_total.values, lw=0.4, alpha=0.4, label="daily")
ax.plot(daily_total.rolling(7).mean(), lw=1.5, label="7-day rolling mean")
ax.set_title("Systemwide L station entries")
ax.set_ylabel("entries/day")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Features and time-based split

Day-of-week uses the **Socrata convention (0=Sunday … 6=Saturday)** to stay consistent
with the app's live-aggregate fallback.

In [ ]:
df["dow"] = (df["date"].dt.weekday + 1) % 7  # pandas Mon=0 -> Socrata Sun=0
df["month"] = df["date"].dt.month
df["week"] = df["date"].dt.isocalendar().week.astype(int)

holidays = USFederalHolidayCalendar().holidays(start=df["date"].min(), end=df["date"].max())
df["is_holiday"] = df["date"].isin(holidays).astype(int)

train = df[df["date"] < SPLIT_DATE].copy()
test = df[df["date"] >= SPLIT_DATE].copy()
print(f"train: {len(train):,} rows (< {SPLIT_DATE}) | test: {len(test):,} rows")

## 4. Baseline: per-station day-of-week mean

The model only earns its place if it beats this.

In [ ]:
baseline = (
    train.groupby(["station_id", "dow"])["rides"].mean().rename("pred").reset_index()
)
test_b = test.merge(baseline, on=["station_id", "dow"], how="left").dropna(subset=["pred"])

def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    mape = (np.abs(y_true - y_pred) / np.clip(y_true, 1, None)).mean() * 100
    return {"model": label, "MAE (riders)": round(mae, 1), "MAPE (%)": round(mape, 1)}

results = [evaluate(test_b["rides"], test_b["pred"], "Baseline: station x dow mean")]
results[-1]

## 5. Gradient boosting

Station is encoded ordinally and declared categorical so the trees can split on it natively.

In [ ]:
FEATURES = ["station_id", "dow", "month", "week", "is_holiday"]

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train = train[FEATURES].copy()
X_test = test[FEATURES].copy()
X_train["station_id"] = encoder.fit_transform(X_train[["station_id"]])
X_test["station_id"] = encoder.transform(X_test[["station_id"]])

model = HistGradientBoostingRegressor(
    max_iter=400, categorical_features=[0], random_state=42
)
model.fit(X_train, train["rides"])

test = test.assign(pred=model.predict(X_test))
results.append(evaluate(test["rides"], test["pred"], "HistGradientBoosting"))
pd.DataFrame(results)

The boosted model's edge over the baseline comes almost entirely from month/seasonality
and holidays — the station × day-of-week interaction is already captured by the baseline.
If the improvement is small, that is itself a finding worth reporting: most of the signal
in daily ridership is just *which station* and *which weekday*.

In [ ]:
# Predicted vs actual on the test period: one busy station, one quiet one
mean_rides = test.groupby("station_id")["rides"].mean()
examples = [mean_rides.idxmax(), mean_rides.idxmin()]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, sid in zip(axes, examples):
    sub = test[test["station_id"] == sid].sort_values("date")
    name = sub["stationname"].iloc[0]
    ax.plot(sub["date"], sub["rides"].rolling(7).mean(), label="actual (7d avg)")
    ax.plot(sub["date"], sub["pred"].rolling(7).mean(), label="predicted (7d avg)")
    ax.set_title(f"{name} (station {sid})")
    ax.set_ylabel("entries/day")
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Export the lookup the app consumes

One prediction per station × day-of-week × month (non-holiday). Tiers are assigned
**within each station's own week** — 3,000 entries is busy for a small station and dead
for Lake/State, so absolute thresholds would mislabel both.

In [ ]:
stations = sorted(df["station_id"].unique())
grid = pd.MultiIndex.from_product(
    [stations, range(7), range(1, 13)], names=["station_id", "dow", "month"]
).to_frame(index=False)
grid["week"] = ((grid["month"] - 1) * 4.35 + 2).astype(int)  # mid-month week number
grid["is_holiday"] = 0

X_grid = grid[FEATURES].copy()
X_grid["station_id"] = encoder.transform(X_grid[["station_id"]])
grid["pred_rides"] = model.predict(X_grid).clip(min=0).round(1)

pct = grid.groupby(["station_id", "month"])["pred_rides"].rank(pct=True)
grid["tier"] = pd.cut(
    pct, bins=[0, 1/3, 2/3, 1], labels=["Quiet", "Moderate", "Busy"], include_lowest=True
).astype(str)

grid[["station_id", "dow", "month", "pred_rides", "tier"]].to_csv(LOOKUP_PATH, index=False)
print(f"wrote {len(grid):,} rows to {LOOKUP_PATH}")
grid.head(10)

## Limitations

- **Daily granularity.** No time-of-day signal; a "Busy" Saturday at a Loop station may still be empty at 8am.
- **Entries only.** The dataset counts turnstile entries, not platform crowding or train load.
- **Station-relative tiers.** "Busy" means busy *for that station*; comparisons across stations should use the raw prediction.
- **No special events.** Cubs games, festivals, and parades move specific days far off the seasonal pattern; an event-calendar feature is the natural next step.
- The most recent months in the dataset lag publication by a few weeks.